# 概率校准与业务阈值

**面试回答：**排序分数不等于概率；校准比较预测分桶与实际发生率，阈值由漏报、误报和人工容量的成本决定。

## 真实案例

退款模型预测 10 笔订单风险，漏掉退款成本 10 元，人工复核成本 1 元。

In [1]:
import numpy as np  # 导入 NumPy 计算校准与成本。
order=np.array(['R01','R02','R03','R04','R05','R06','R07','R08','R09','R10'])  # 构造订单编号。
p=np.array([.08,.12,.22,.35,.42,.55,.62,.74,.86,.92])  # 记录模型预测概率。
y=np.array([0,0,0,1,0,0,1,1,1,1])  # 记录延迟到达的退款标签。
print('订单 | 预测概率 | 真实退款')  # 输出账本表头。
for a,b,c in zip(order,p,y):  # 逐条展示样本。
    print(a,b,c)  # 输出一条订单。

订单 | 预测概率 | 真实退款
R01 0.08 0
R02 0.12 0
R03 0.22 0
R04 0.35 1
R05 0.42 0
R06 0.55 0
R07 0.62 1
R08 0.74 1
R09 0.86 1
R10 0.92 1


## Baseline / 基线

固定 0.5 阈值忽略不同业务成本。

In [2]:
base=(p>=.5).astype(int)  # 用固定阈值生成审核决策。
base_cost=int(np.sum((base==0)&(y==1))*10+np.sum((base==1)&(y==0)))  # 计算漏报和复核成本。
print('0.5阈值审核量/成本=',int(base.sum()),base_cost)  # 输出基线。

0.5阈值审核量/成本= 5 11


In [3]:
bins=[(0,.3),(.3,.6),(.6,1.01)]  # 定义三个概率分桶。
for low,high in bins:  # 逐桶检查可靠性。
    mask=(p>=low)&(p<high)  # 选出当前概率桶。
    print('桶',low,high,'预测均值=',round(float(p[mask].mean()),3),'实际率=',round(float(y[mask].mean()),3),'样本=',int(mask.sum()))  # 输出校准中间量。
thresholds=np.array([.2,.35,.5,.65,.8])  # 枚举可选审核阈值。
costs=[]  # 保存每个阈值成本。
for threshold in thresholds:  # 按成本评估阈值。
    audit=p>=threshold  # 生成审核队列。
    costs.append(int(np.sum((~audit)&(y==1))*10+np.sum(audit&(y==0))))  # 计算总业务成本。
best_threshold=float(thresholds[int(np.argmin(costs))])  # 选择最低成本阈值。
print('阈值/成本=',list(zip(thresholds.tolist(),costs)))  # 输出阈值搜索表。

桶 0 0.3 预测均值= 0.14 实际率= 0.0 样本= 3
桶 0.3 0.6 预测均值= 0.44 实际率= 0.333 样本= 3
桶 0.6 1.01 预测均值= 0.785 实际率= 1.0 样本= 4
阈值/成本= [(0.2, 3), (0.35, 2), (0.5, 11), (0.65, 20), (0.8, 30)]


## 结果解读

校准桶要求预测均值接近实际率；本例阈值由成本选出，不由 0.5 习惯决定。真实校准器必须在独立验证集拟合。

In [4]:
best_audit=p>=best_threshold  # 用最低成本阈值生成最终队列。
print('最优阈值/审核量/成本=',best_threshold,int(best_audit.sum()),min(costs))  # 输出最终业务决策。
print('生产差距：需独立校准集、Brier/可靠性曲线、分群校准与审核容量监控。')  # 说明生产边界。
print('教学样本很小，分桶差异不能外推为线上校准结论。')  # 说明实验边界。

最优阈值/审核量/成本= 0.35 7 2
生产差距：需独立校准集、Brier/可靠性曲线、分群校准与审核容量监控。
教学样本很小，分桶差异不能外推为线上校准结论。


## 失败案例与修复

若在最终测试集反复调阈值，测试集已参与决策；修复是划分训练、校准、测试三段并冻结阈值。

In [5]:
test_optimal=float(thresholds[int(np.argmin(costs))])  # 故意展示在同一标签集挑阈值的错误动作。
print('失败：在报告集挑出的阈值=',test_optimal)  # 输出泄漏动作。
print('修复：在独立校准窗口选择阈值，最终测试只报告一次。')  # 输出修复原则。
print('概率校准不等于业务公平或因果概率。')  # 说明边界。

失败：在报告集挑出的阈值= 0.35
修复：在独立校准窗口选择阈值，最终测试只报告一次。
概率校准不等于业务公平或因果概率。


In [6]:
assert len(order)>=5  # 保护样本数。
assert min(costs)<=base_cost  # 保护成本选阈值不差于固定阈值。
assert 0<best_threshold<1  # 保护阈值合法。
assert int(best_audit.sum())>0  # 保护存在实际审核队列。